In [ ]:
import os
import sys

# Make the project's src package available from the notebooks folder.
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data, wearables, baselines, evaluation

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
SEED = 0

In [ ]:
# Summarize source readings before outlier filtering or daily aggregation.
# Treat zeros as missing for this analysis; raw_signals retains the original readings.
# Statistics skip missing values. SD is the sample standard deviation.
# nightly_temperature is average nightly skin temperature in Celsius (data/raw/README.txt).
# Missingness is within existing source rows, NOT days with no recorded row.
# Sources have different sampling frequencies; n_records does not mean days.
signal_sources = {
    "resting_hr": ("resting_heart_rate.csv", "value"),
    "skin_temp": ("computed_temperature.csv", "nightly_temperature"),
    "hrv_rmssd": ("heart_rate_variability_details.csv", "rmssd"),
    "breathing_rate": ("respiratory_rate_summary.csv", "full_sleep_breathing_rate"),
}
raw_signals = {}
summary_rows = []
for signal, (filename, column) in signal_sources.items():
    raw = pd.read_csv(data.data_path(filename))
    raw_signals[signal] = raw
    values = raw[column].replace(0, np.nan)
    summary_rows.append({
        "signal": signal,
        "n_records": len(values),
        "min": values.min(),
        "max": values.max(),
        "median": values.median(),
        "mean": values.mean(),
        "sd": values.std(),
        "n_missing": values.isna().sum(),
        "pct_missing": 100 * values.isna().mean(),
    })
signal_summary = pd.DataFrame(summary_rows).set_index("signal")
signal_summary.round(2)

## Recording frequency and coverage
Compare source records per day and missing-day coverage for each signal.


In [ ]:
# Look for per-feature coverage against unique days in diary.
# A day is covered if it has at least one nonzero, nonmissing reading.
# Use sleep END day for temperature; source day_in_study for other signals.
reference_days = data.load_diary()[wearables.KEY].drop_duplicates()

def gap_counts(expected_days, valid_days):
    missing = expected_days - valid_days
    counts = {
        "reference_days": len(expected_days),
        "covered_days": len(expected_days & valid_days),
        "missing_days": len(missing),
        "internal_gap_days": 0,
        "before_first_valid": 0,
        "after_last_valid": 0,
        "no_valid_readings_days": 0,
    }
    if not valid_days:
        counts["no_valid_readings_days"] = len(missing)
    else:
        first, last = min(valid_days), max(valid_days)
        counts["before_first_valid"] = sum(day < first for day in missing)
        counts["after_last_valid"] = sum(day > last for day in missing)
        counts["internal_gap_days"] = sum(first < day < last for day in missing)
    return counts

coverage_rows = []
for signal, (_, column) in signal_sources.items():
    raw = raw_signals[signal]
    day_column = "sleep_end_day_in_study" if signal == "skin_temp" else "day_in_study"
    valid = raw.loc[raw[column].notna() & raw[column].ne(0)]
    valid_by_person = {
        key: set(group[day_column].dropna())
        for key, group in valid.groupby(["id", "study_interval"])
    }
    for (participant, interval), group in reference_days.groupby(["id", "study_interval"]):
        counts = gap_counts(
            set(group.day_in_study),
            valid_by_person.get((participant, interval), set()),
        )
        coverage_rows.append({
            "signal": signal, "id": participant, "study_interval": interval, **counts,
        })

# Keep participant-level results available for later inspection, without saving a file.
coverage_by_participant = pd.DataFrame(coverage_rows)
coverage_by_participant["pct_missing"] = (
    100 * coverage_by_participant.missing_days / coverage_by_participant.reference_days
).round(2)
count_columns = list(counts)
coverage_summary = coverage_by_participant.groupby(["signal", "study_interval"])[count_columns].sum()
coverage_summary["pct_missing"] = (
    100 * coverage_summary.missing_days / coverage_summary.reference_days
).round(2)
coverage_summary

Compare missing calendar days separately from missing values in recorded rows.

In [ ]:
# Count nonzero, nonmissing source records on days with at least one valid reading.
# Each participant-day has equal weight; days with no valid readings are excluded.
# Use all recorded days, not just diary days; no deduplication or outlier filtering.
# Temperature uses sleep end day; other signals retain their source day.
daily_record_counts = {}
record_count_tables = []
for signal, (_, column) in signal_sources.items():
    raw = raw_signals[signal]
    day_column = "sleep_end_day_in_study" if signal == "skin_temp" else "day_in_study"
    valid = raw.loc[raw[column].notna() & raw[column].ne(0)]
    counts = valid.groupby(["id", "study_interval", day_column]).size()
    daily_record_counts[signal] = counts.rename("n_records")
    summary = counts.groupby(level="study_interval").agg(
        nonmissing_days="size",
        median_records_per_day="median",
        mean_records_per_day="mean",
    ).reset_index()
    summary["signal"] = signal
    record_count_tables.append(summary)
records_per_day_summary = pd.concat(record_count_tables, ignore_index=True)
records_per_day_summary.set_index(["signal", "study_interval"]).round(2)

Compare recording frequency across intervals before choosing daily aggregation.


In [ ]:
# Per-participant extrema across all valid raw readings (no daily aggregation).
# Pooling adds a later period as well as more readings/day: this is not a causal test.
hr = raw_signals["resting_hr"]
hr = hr.loc[hr.value.notna() & hr.value.ne(0)]
hr_interval1 = hr.loc[hr.study_interval == 2022].groupby("id").value.agg(["min", "max"])
hr_pooled = hr.loc[hr.study_interval.isin([2022, 2024])].groupby("id").value.agg(["min", "max"])
participant_ids = hr_pooled.index.sort_values()
hr_interval1 = hr_interval1.reindex(participant_ids)
hr_pooled = hr_pooled.reindex(participant_ids)

x = np.arange(len(participant_ids))
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, statistic in zip(axes, ["min", "max"]):
    ax.scatter(x - 0.12, hr_interval1[statistic], color="C0", marker="o",
               label="Interval 1 only", s=28)
    ax.scatter(x + 0.12, hr_pooled[statistic], color="C1", marker="x",
               label="Intervals 1 + 2 pooled", s=32)
    ax.set_xticks(x, participant_ids.astype(str), rotation=90, fontsize=8)
    ax.set_xlabel("Participant ID")
    ax.set_title(f"{statistic.capitalize()} resting HR per participant")
    ax.legend(fontsize=9)
axes[0].set_ylabel("Resting heart rate (bpm)")
fig.suptitle("All valid raw HR readings: Interval 1 vs both intervals")
fig.tight_layout()
# Participants without Interval 2 data have identical values in both series.
# Pooling can only lower/retain the minimum and raise/retain the maximum.
plt.show()



In [ ]:
# Select participants sharing the exact Interval 2 maximum (no rounding for selection).
hr_raw = raw_signals["resting_hr"]
hr_interval2 = hr_raw.loc[
    hr_raw.study_interval.eq(2024) & hr_raw.value.notna() & hr_raw.value.ne(0)
]
shared_max_hr = hr_interval2.value.max()
shared_max_ids = hr_interval2.loc[hr_interval2.value.eq(shared_max_hr), "id"].unique()
selected_hr_interval2 = hr_interval2.loc[hr_interval2.id.isin(shared_max_ids)].copy()

# Summarize ALL their valid Interval 2 readings, not just records at the maximum.
# Average = mean. SD is sample SD; each raw reading has equal weight, not each day.
shared_max_hr_summary = selected_hr_interval2.groupby("id").value.agg(
    min="min", max="max", mean="mean", sd="std"
)
print(f"{len(shared_max_ids)} participants share the maximum of {shared_max_hr:.7f} bpm")
shared_max_hr_summary.round(2)

In [ ]:
# Audit exact HR-error pairs shared by DIFFERENT participants within each interval.
# Match full stored precision; round only for display. Shared does not prove erroneous.
# Exclude zero/missing HR; count missing error separately (zero error is retained).
from IPython.display import display

hr_for_pairs = raw_signals["resting_hr"]
hr_for_pairs = hr_for_pairs.loc[hr_for_pairs.value.notna() & hr_for_pairs.value.ne(0)]
pair_counts = (
    hr_for_pairs.dropna(subset=["error"])
    .groupby(["study_interval", "value", "error"])
    .agg(n_records=("id", "size"), n_participants=("id", "nunique"))
    .reset_index()
)
shared_pairs = pair_counts.loc[pair_counts.n_participants > 1].copy()

pair_summary_rows = []
for interval, readings in hr_for_pairs.groupby("study_interval"):
    pairs = pair_counts.loc[pair_counts.study_interval == interval]
    shared = shared_pairs.loc[shared_pairs.study_interval == interval]
    compared_records = int(pairs.n_records.sum())
    shared_records = int(shared.n_records.sum())
    pair_summary_rows.append({
        "study_interval": interval,
        "participants": readings.id.nunique(),
        "valid_hr_records": len(readings),
        "missing_error_records": readings.error.isna().sum(),
        "records_compared": compared_records,
        "unique_pairs": len(pairs),
        "pairs_shared_across_people": len(shared),
        "records_in_shared_pairs": shared_records,
        "pct_records_in_shared_pairs": (
            100 * shared_records / compared_records if compared_records else np.nan
        ),
    })
hr_error_summary = pd.DataFrame(pair_summary_rows).set_index("study_interval")
display(hr_error_summary.round(2))

# Top 10 shared pairs per interval. n_records can include repeated days per person;
# n_participants counts distinct people, regardless of when their records occurred.
top_shared_pairs = (
    shared_pairs.sort_values(
        ["study_interval", "n_participants", "n_records"],
        ascending=[True, False, False],
    )
    .groupby("study_interval").head(10)
    .set_index(["study_interval", "value", "error"])
)
display(top_shared_pairs)

# Inspect error values attached specifically to the shared Interval 2 maximum.
interval2_pairs = pair_counts.loc[pair_counts.study_interval == 2024]
maximum_pairs = interval2_pairs.loc[
    interval2_pairs.value.eq(interval2_pairs.value.max())
].set_index(["study_interval", "value", "error"])
display(maximum_pairs)

## Feature Independence

See how assocaited features are for later model fitting with z-score variation across 3 randomly selected individuals.

In [ ]:
# Exploratory comparison only: Interval 1 while Interval 2 HR is under review.
# Daily medians retain all nonzero/nonmissing readings; no outlier filtering.
# Temperature is assigned to sleep end day. HRV uses source day_in_study
# (not shifted to waking day); precise night alignment still needs validation.
plot_interval = 2022
daily_series = []
for signal, (_, column) in signal_sources.items():
    raw = raw_signals[signal]
    day_column = "sleep_end_day_in_study" if signal == "skin_temp" else "day_in_study"
    valid = raw.loc[
        raw.study_interval.eq(plot_interval) & raw[column].notna() & raw[column].ne(0)
    ]
    series = valid.groupby(["id", day_column])[column].median().rename(signal)
    series.index = series.index.set_names(["id", "day_in_study"])
    daily_series.append(series)
daily_for_plot = pd.concat(daily_series, axis=1).sort_index()

# Random choice, not hand-picked trends; require variation in all four signals.
signal_spreads = daily_for_plot.groupby(level="id").std()
eligible_ids = signal_spreads.index[signal_spreads.gt(0).all(axis=1)].sort_values()
if len(eligible_ids) < 3:
    raise ValueError("Fewer than three participants have varying data for all four signals.")
selected_plot_ids = np.random.default_rng(SEED).choice(eligible_ids, size=3, replace=False)

labels = ["Resting HR", "Skin temperature", "HRV (RMSSD)", "Breathing rate"]
styles = ["-", "--", "-.", ":"]
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True, sharey=True)
standardized_plot_data = {}
for ax, participant in zip(axes, selected_plot_ids):
    daily = daily_for_plot.xs(participant, level="id")
    # Insert absent days so lines do not bridge gaps.
    daily = daily.reindex(range(int(daily.index.min()), int(daily.index.max()) + 1))
    # Full-period standardization is for visualization, NOT a predictive feature.
    z = (daily - daily.mean()) / daily.std().replace(0, np.nan)
    standardized_plot_data[participant] = z
    for i, (signal, label, style) in enumerate(zip(signal_sources, labels, styles)):
        ax.plot(z.index, z[signal], label=label, color=f"C{i}",
                linestyle=style, linewidth=1.3, marker=".", markersize=3)
    ax.axhline(0, color="0.5", linewidth=0.7)
    ax.set_title(f"Participant {participant}")
    ax.set_ylabel("Within-person SD")
axes[0].legend(ncol=4, fontsize=9, loc="upper right")
axes[-1].set_xlabel("Day in study (not cycle day)")
fig.suptitle(f"Wearable signals together — Interval {plot_interval}, daily medians")
fig.tight_layout()
plt.show()

Pearson correlations for 6 pairs of each 4 features to look for statistical significance between features.

In [ ]:

from itertools import combinations
from IPython.display import display

if plot_interval != 2022:
    raise ValueError("Set plot_interval = 2022 and rerun the preceding cell first.")

correlation_rows = []
for participant, daily in daily_for_plot.groupby(level="id"):
    for signal_a, signal_b in combinations(signal_sources, 2):
        paired = daily[[signal_a, signal_b]].dropna()
        n_days = len(paired)
        r = np.nan
        status = "ok"
        if n_days < 3:
            status = "fewer than 3 paired days"
        elif paired.nunique().min() < 2:
            status = "constant signal on paired days"
        else:
            r = paired[signal_a].corr(paired[signal_b], method="pearson")
        correlation_rows.append({
            "id": participant,
            "pair": f"{signal_a} / {signal_b}",
            "paired_days": n_days,
            "pearson_r": r,
            "status": status,
        })

correlations_by_participant = pd.DataFrame(correlation_rows)
# Three days is a calculation floor, not evidence of a reliable relationship.
correlation_summary = correlations_by_participant.groupby("pair").agg(
    participants_with_r=("pearson_r", "count"),
    median_r=("pearson_r", "median"),
    min_r=("pearson_r", "min"),
    max_r=("pearson_r", "max"),
    median_paired_days=("paired_days", "median"),
    min_paired_days=("paired_days", "min"),
)
display(correlation_summary.round(3))

# Show only correlations; paired-day counts remain in correlations_by_participant.
correlation_table = correlations_by_participant.pivot(
    index="id", columns="pair", values="pearson_r"
)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(correlation_table.round(3))

Inspect the pairwise correlation summaries locally; dataset-derived findings belong in private notes.


## Additional Features
Add features generated from raw row outputs. For example, [many existing studies](https://www.nature.com/articles/s44294-025-00078-8) use 6-day window median, min, max, standard deviations, and slopes to indicate if the signal has been consistenly high or low or consistently changing over the last several days. 
Next, we will add in 3-day windows to look for predictive shorter term changes.
In addition, we will do abalation testing to see which column groups are best.

Show line chart of a participant's raw timeseries data, visually compare to 6-day median, min, max, SD, and slope. One for each feature.


In [ ]:
# Six-day features for one reproducibly random Interval 1 participant.
# Uses the daily medians prepared above, not individual raw records.
# Exploratory only: source-day/night alignment is still provisional.
if plot_interval != 2022:
    raise ValueError("Set plot_interval = 2022 and rerun the daily-signal cell first.")
window_days = 6
min_valid_days = 6  # Require a complete six-day window; do not fill gaps.
sample_days = 60

# Prefer participants with a 60-calendar-day span and at least six readings per signal.
participant_spans = daily_for_plot.reset_index().groupby("id").day_in_study.agg(["min", "max"])
enough_readings = daily_for_plot.groupby(level="id").count().ge(min_valid_days).all(axis=1)
candidate_ids = participant_spans.index[enough_readings]
long_ids = candidate_ids[
    (participant_spans.loc[candidate_ids, "max"] - participant_spans.loc[candidate_ids, "min"] + 1) >= sample_days
]
if len(long_ids):
    candidate_ids = long_ids
if not len(candidate_ids):
    raise ValueError("No participant has enough readings for this plot.")
rolling_plot_id = np.random.default_rng(SEED).choice(candidate_ids.sort_values())
participant_daily = daily_for_plot.xs(rolling_plot_id, level="id").sort_index()
participant_daily = participant_daily.reindex(
    range(int(participant_daily.index.min()), int(participant_daily.index.max()) + 1)
)

def calendar_day_slope(values):
    # Fit a straight line using actual study-day positions, not compressed reading order.
    valid = values.dropna()
    if len(valid) < 2:
        return np.nan
    days = valid.index.to_numpy(dtype=float)
    return np.polyfit(days - days[0], valid.to_numpy(), 1)[0]

rolling_feature_examples = {}
for signal in signal_sources:
    daily = participant_daily[signal]
    window = daily.rolling(window_days, min_periods=min_valid_days)
    rolling_feature_examples[signal] = pd.DataFrame({
        "daily": daily,
        "median": window.median(),
        "sd": window.std(),
        "min": window.min(),
        "max": window.max(),
        "slope": window.apply(calendar_day_slope, raw=False),
    }).iloc[:sample_days]

# Levels and SD share the left axis; slope has units per day on the right.
signal_labels = {
    "resting_hr": ("Resting HR", "bpm"),
    "skin_temp": ("Skin temperature", "°C"),
    "hrv_rmssd": ("HRV (RMSSD)", "ms"),
    "breathing_rate": ("Breathing rate", "breaths/min"),
}
stat_styles = {
    "daily": ("C0", "-", "Daily value"),
    "median": ("C1", "-", "6-day median"),
    "sd": ("C2", ":", "6-day SD"),
    "min": ("C3", "--", "6-day min"),
    "max": ("C4", "-.", "6-day max"),
}
fig, axes = plt.subplots(4, 1, figsize=(14, 13), sharex=True)
for ax, (signal, (label, unit)) in zip(axes, signal_labels.items()):
    example = rolling_feature_examples[signal]
    for statistic, (color, style, legend_label) in stat_styles.items():
        ax.plot(example.index, example[statistic], color=color, linestyle=style,
                linewidth=1.4, label=legend_label,
                marker="." if statistic == "daily" else None, markersize=3)
    slope_axis = ax.twinx()
    slope_axis.grid(False)
    slope_axis.plot(example.index, example.slope, color="C5", linestyle="--",
                    linewidth=1.4, label="6-day slope (right axis)")
    slope_axis.set_ylabel(f"Slope ({unit} per day)")
    ax.set_ylabel(unit)
    ax.set_title(label, loc="left")
    if ax is axes[0]:
        lines, names = ax.get_legend_handles_labels()
        slope_lines, slope_names = slope_axis.get_legend_handles_labels()
        ax.legend(lines + slope_lines, names + slope_names,
                  ncol=3, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, 1.45))
axes[-1].set_xlabel("Day in study (not cycle day)")
shown_days = len(next(iter(rolling_feature_examples.values())))
fig.suptitle(
    f"Participant {rolling_plot_id} — Interval 1 — first {shown_days} calendar days\n"
    "Trailing windows include today; six valid days required; missing data are not interpolated",
    y=0.995,
)
fig.tight_layout(rect=[0, 0, 1, 0.94], h_pad=2)
plt.show()

Now do the same for a 3-day window.

In [ ]:
# Three-day version: run the six-day cell first to reuse the same participant and days.
three_day_feature_examples = {}
for signal in signal_sources:
    daily = participant_daily[signal]
    window = daily.rolling(3, min_periods=3)
    three_day_feature_examples[signal] = pd.DataFrame({
        "daily": daily,
        "median": window.median(),
        "sd": window.std(),
        "min": window.min(),
        "max": window.max(),
        "slope": window.apply(calendar_day_slope, raw=False),
    }).iloc[:sample_days]

three_day_styles = {
    "daily": ("C0", "-", "Daily value"),
    "median": ("C1", "-", "3-day median"),
    "sd": ("C2", ":", "3-day SD"),
    "min": ("C3", "--", "3-day min"),
    "max": ("C4", "-.", "3-day max"),
}
fig, axes = plt.subplots(4, 1, figsize=(14, 13), sharex=True)
for ax, (signal, (label, unit)) in zip(axes, signal_labels.items()):
    example = three_day_feature_examples[signal]
    for statistic, (color, style, legend_label) in three_day_styles.items():
        ax.plot(example.index, example[statistic], color=color, linestyle=style,
                linewidth=1.4, label=legend_label,
                marker="." if statistic == "daily" else None, markersize=3)
    slope_axis = ax.twinx()
    slope_axis.grid(False)
    slope_axis.plot(example.index, example.slope, color="C5", linestyle="--",
                    linewidth=1.4, label="3-day slope (right axis)")
    slope_axis.set_ylabel(f"Slope ({unit} per day)")
    ax.set_ylabel(unit)
    ax.set_title(label, loc="left")
    if ax is axes[0]:
        lines, names = ax.get_legend_handles_labels()
        slope_lines, slope_names = slope_axis.get_legend_handles_labels()
        ax.legend(lines + slope_lines, names + slope_names,
                  ncol=3, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, 1.45))
axes[-1].set_xlabel("Day in study (not cycle day)")
shown_days = len(next(iter(three_day_feature_examples.values())))
fig.suptitle(
    f"Participant {rolling_plot_id} — Interval 1 — first {shown_days} calendar days\n"
    "Trailing windows include today; three valid days required; missing data are not interpolated",
    y=0.995,
)
fig.tight_layout(rect=[0, 0, 1, 0.94], h_pad=2)
plt.show()